<a href="https://colab.research.google.com/github/Harshada900/pyspark-practise/blob/main/13_Window_Function_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
import pyspark.sql.functions as f

spark = SparkSession.builder \
    .appName("WindowFunctions") \
    .config("spark.sql.shuffle.partitions", "5") \
    .getOrCreate()

data = [("Raj",   "Engineering", 50000),
        ("Amit",  "Engineering", 60000),
        ("John",  "Engineering", 90000),
        ("Priya", "Marketing",   70000),
        ("Karan", "Marketing",   45000),
        ("Sara",  "HR",          80000),
        ("Neha",  "HR",          80000),   # same salary as Sara
        ("Raj2",  "HR",          55000)]

columns = ["name", "department", "salary"]
df = spark.createDataFrame(data, columns)
df.show()

+-----+-----------+------+
| name| department|salary|
+-----+-----------+------+
|  Raj|Engineering| 50000|
| Amit|Engineering| 60000|
| John|Engineering| 90000|
|Priya|  Marketing| 70000|
|Karan|  Marketing| 45000|
| Sara|         HR| 80000|
| Neha|         HR| 80000|
| Raj2|         HR| 55000|
+-----+-----------+------+



In [3]:
from pyspark.sql.window import Window

#define the window

window = Window.partitionBy("department").orderBy("salary")

#here window is window specification, it doesnt store data,

## **1. Aggregation with Window — row level + group level together**

In [4]:
window_dept = Window.partitionBy("department")
df.withColumn("max_salary", f.max("salary").over(window_dept))\
.withColumn("min_salary", f.min("salary").over(window_dept))\
.withColumn("avg_salary", f.avg("salary").over(window_dept))\
.withColumn("total_salary", f.sum("salary").over(window_dept)).show()       #keeps original rows!

+-----+-----------+------+----------+----------+-----------------+------------+
| name| department|salary|max_salary|min_salary|       avg_salary|total_salary|
+-----+-----------+------+----------+----------+-----------------+------------+
|  Raj|Engineering| 50000|     90000|     50000|66666.66666666667|      200000|
| Amit|Engineering| 60000|     90000|     50000|66666.66666666667|      200000|
| John|Engineering| 90000|     90000|     50000|66666.66666666667|      200000|
| Sara|         HR| 80000|     80000|     55000|71666.66666666667|      215000|
| Neha|         HR| 80000|     80000|     55000|71666.66666666667|      215000|
| Raj2|         HR| 55000|     80000|     55000|71666.66666666667|      215000|
|Priya|  Marketing| 70000|     70000|     45000|          57500.0|      115000|
|Karan|  Marketing| 45000|     70000|     45000|          57500.0|      115000|
+-----+-----------+------+----------+----------+-----------------+------------+



## **2. RANK — Ranking Within a Group**

In [7]:
window_rank = Window.partitionBy("department").orderBy(f.col("salary").desc()) #orderby is req bcz we need to tell spark on what basis it should rank rows..

df.withColumn("rank", f.rank().over(window_rank)).show()

+-----+-----------+------+----+
| name| department|salary|rank|
+-----+-----------+------+----+
| John|Engineering| 90000|   1|
| Amit|Engineering| 60000|   2|
|  Raj|Engineering| 50000|   3|
| Sara|         HR| 80000|   1|
| Neha|         HR| 80000|   1|
| Raj2|         HR| 55000|   3|
|Priya|  Marketing| 70000|   1|
|Karan|  Marketing| 45000|   2|
+-----+-----------+------+----+



## **3. DENSE_RANK — Ranking Without Gaps**

In [9]:
df.withColumn("dense_rank", f.dense_rank().over(window_rank)).show()

+-----+-----------+------+----------+
| name| department|salary|dense_rank|
+-----+-----------+------+----------+
| John|Engineering| 90000|         1|
| Amit|Engineering| 60000|         2|
|  Raj|Engineering| 50000|         3|
| Sara|         HR| 80000|         1|
| Neha|         HR| 80000|         1|
| Raj2|         HR| 55000|         2|
|Priya|  Marketing| 70000|         1|
|Karan|  Marketing| 45000|         2|
+-----+-----------+------+----------+



## **4. ROW_NUMBER — Unique Number for Every Row**

In [11]:
df.withColumn("row_num", f.row_number().over(window_rank)).show()

+-----+-----------+------+-------+
| name| department|salary|row_num|
+-----+-----------+------+-------+
| John|Engineering| 90000|      1|
| Amit|Engineering| 60000|      2|
|  Raj|Engineering| 50000|      3|
| Sara|         HR| 80000|      1|
| Neha|         HR| 80000|      2|
| Raj2|         HR| 55000|      3|
|Priya|  Marketing| 70000|      1|
|Karan|  Marketing| 45000|      2|
+-----+-----------+------+-------+



## **5. LAG — Look at Previous Row's Value**

In [12]:
window_lag = Window.partitionBy("department").orderBy("salary")
df.withColumn("lag", f.lag("salary", 1).over(window_lag)).show() #1 indicates how many rows back

+-----+-----------+------+-----+
| name| department|salary|  lag|
+-----+-----------+------+-----+
|  Raj|Engineering| 50000| NULL|
| Amit|Engineering| 60000|50000|
| John|Engineering| 90000|60000|
| Raj2|         HR| 55000| NULL|
| Sara|         HR| 80000|55000|
| Neha|         HR| 80000|80000|
|Karan|  Marketing| 45000| NULL|
|Priya|  Marketing| 70000|45000|
+-----+-----------+------+-----+



In [13]:
df.withColumn("lag", f.lag("salary", 2).over(window_lag)).show()

+-----+-----------+------+-----+
| name| department|salary|  lag|
+-----+-----------+------+-----+
|  Raj|Engineering| 50000| NULL|
| Amit|Engineering| 60000| NULL|
| John|Engineering| 90000|50000|
| Raj2|         HR| 55000| NULL|
| Sara|         HR| 80000| NULL|
| Neha|         HR| 80000|55000|
|Karan|  Marketing| 45000| NULL|
|Priya|  Marketing| 70000| NULL|
+-----+-----------+------+-----+



## **Calculate Salary Difference from Previous Employee**

In [15]:
df.withColumn("lag", f.lag("salary", 1).over(window_lag)).withColumn("salary_diff", f.col("salary")- f.lag("salary", 1).over(window_lag)).show()

+-----+-----------+------+-----+-----------+
| name| department|salary|  lag|salary_diff|
+-----+-----------+------+-----+-----------+
|  Raj|Engineering| 50000| NULL|       NULL|
| Amit|Engineering| 60000|50000|      10000|
| John|Engineering| 90000|60000|      30000|
| Raj2|         HR| 55000| NULL|       NULL|
| Sara|         HR| 80000|55000|      25000|
| Neha|         HR| 80000|80000|          0|
|Karan|  Marketing| 45000| NULL|       NULL|
|Priya|  Marketing| 70000|45000|      25000|
+-----+-----------+------+-----+-----------+



## **6. LEAD — Look at Next Row's Value**

In [16]:
df.withColumn("next_salary", f.lead("salary", 1).over(window_lag)).show()

+-----+-----------+------+-----------+
| name| department|salary|next_salary|
+-----+-----------+------+-----------+
|  Raj|Engineering| 50000|      60000|
| Amit|Engineering| 60000|      90000|
| John|Engineering| 90000|       NULL|
| Raj2|         HR| 55000|      80000|
| Sara|         HR| 80000|      80000|
| Neha|         HR| 80000|       NULL|
|Karan|  Marketing| 45000|      70000|
|Priya|  Marketing| 70000|       NULL|
+-----+-----------+------+-----------+



## **Window Functions in Spark SQL**

In [17]:
df.createOrReplaceTempView("employees")

spark.sql("""
select name, department, salary,
rank() over(partition by department order by salary desc) as rank,
lag(salary, 1) over(partition by department order by salary desc) as prev_salary,
avg(salary) over(partition by department) as avg_salary
from employees

""").show()

+-----+-----------+------+----+-----------+-----------------+
| name| department|salary|rank|prev_salary|       avg_salary|
+-----+-----------+------+----+-----------+-----------------+
| John|Engineering| 90000|   1|       NULL|66666.66666666667|
| Amit|Engineering| 60000|   2|      90000|66666.66666666667|
|  Raj|Engineering| 50000|   3|      60000|66666.66666666667|
| Sara|         HR| 80000|   1|       NULL|71666.66666666667|
| Neha|         HR| 80000|   1|      80000|71666.66666666667|
| Raj2|         HR| 55000|   3|      80000|71666.66666666667|
|Priya|  Marketing| 70000|   1|       NULL|          57500.0|
|Karan|  Marketing| 45000|   2|      70000|          57500.0|
+-----+-----------+------+----+-----------+-----------------+



## **Window Without partitionBy**

In [18]:
window = Window.orderBy("salary")

df.withColumn("overall_rank", f.rank().over(window)).show()

+-----+-----------+------+------------+
| name| department|salary|overall_rank|
+-----+-----------+------+------------+
|Karan|  Marketing| 45000|           1|
|  Raj|Engineering| 50000|           2|
| Raj2|         HR| 55000|           3|
| Amit|Engineering| 60000|           4|
|Priya|  Marketing| 70000|           5|
| Sara|         HR| 80000|           6|
| Neha|         HR| 80000|           6|
| John|Engineering| 90000|           8|
+-----+-----------+------+------------+



## **Practice Tasks**

Find top 2 highest paid employees per department

Add a column showing each employee's salary difference from department average

Add a column showing rank of each employee within their department by salary

Using LAG — add a column showing previous employee's salary within same
department (ordered by salary)

Find employees earning more than the average salary of their department

In [30]:
win_dept = Window.partitionBy("department").orderBy(f.desc("salary"))

df.withColumn("2nd_highest_salary", f.dense_rank().over(win_dept)).filter(f.col("2nd_highest_salary")<=2).show()

+-----+-----------+------+------------------+
| name| department|salary|2nd_highest_salary|
+-----+-----------+------+------------------+
| John|Engineering| 90000|                 1|
| Amit|Engineering| 60000|                 2|
| Sara|         HR| 80000|                 1|
| Neha|         HR| 80000|                 1|
| Raj2|         HR| 55000|                 2|
|Priya|  Marketing| 70000|                 1|
|Karan|  Marketing| 45000|                 2|
+-----+-----------+------+------------------+



In [26]:
win_dept1 = Window.partitionBy("department")
df.withColumn("avg_salary_dept", f.round(f.avg("salary").over(win_dept1),0)).withColumn("salary_diff", f.round(f.col("salary")-f.col("avg_salary_dept"))).show()

+-----+-----------+------+---------------+-----------+
| name| department|salary|avg_salary_dept|salary_diff|
+-----+-----------+------+---------------+-----------+
|  Raj|Engineering| 50000|        66667.0|   -16667.0|
| Amit|Engineering| 60000|        66667.0|    -6667.0|
| John|Engineering| 90000|        66667.0|    23333.0|
| Sara|         HR| 80000|        71667.0|     8333.0|
| Neha|         HR| 80000|        71667.0|     8333.0|
| Raj2|         HR| 55000|        71667.0|   -16667.0|
|Priya|  Marketing| 70000|        57500.0|    12500.0|
|Karan|  Marketing| 45000|        57500.0|   -12500.0|
+-----+-----------+------+---------------+-----------+



In [27]:
df.withColumn("rank", f.rank().over(win_dept)).show()

+-----+-----------+------+----+
| name| department|salary|rank|
+-----+-----------+------+----+
| John|Engineering| 90000|   1|
| Amit|Engineering| 60000|   2|
|  Raj|Engineering| 50000|   3|
| Sara|         HR| 80000|   1|
| Neha|         HR| 80000|   1|
| Raj2|         HR| 55000|   3|
|Priya|  Marketing| 70000|   1|
|Karan|  Marketing| 45000|   2|
+-----+-----------+------+----+



In [31]:
win_lag = Window.partitionBy("department").orderBy("salary")
df.withColumn("prev_salary", f.lag("salary", 1).over(win_lag)).show()

+-----+-----------+------+-----------+
| name| department|salary|prev_salary|
+-----+-----------+------+-----------+
|  Raj|Engineering| 50000|       NULL|
| Amit|Engineering| 60000|      50000|
| John|Engineering| 90000|      60000|
| Raj2|         HR| 55000|       NULL|
| Sara|         HR| 80000|      55000|
| Neha|         HR| 80000|      80000|
|Karan|  Marketing| 45000|       NULL|
|Priya|  Marketing| 70000|      45000|
+-----+-----------+------+-----------+



In [29]:
df.withColumn("avg_salary_dept", f.round(f.avg("salary").over(win_dept1),0)).filter(f.col("salary")>f.col("avg_salary_dept")).show()

+-----+-----------+------+---------------+
| name| department|salary|avg_salary_dept|
+-----+-----------+------+---------------+
| John|Engineering| 90000|        66667.0|
| Sara|         HR| 80000|        71667.0|
| Neha|         HR| 80000|        71667.0|
|Priya|  Marketing| 70000|        57500.0|
+-----+-----------+------+---------------+

